# Initial data processing pipeline for ANTIPASTI input

In [2]:
import os
import torch
import matplotlib.pyplot as plt

from antipasti.preprocessing.preprocessing import Preprocessing
from antipasti.utils.torch_utils import create_test_set, save_checkpoint, load_checkpoint, training_routine

In [3]:
# Assign directories

# AF3 directories
AF_INPUT_PATH = '../af3/af_input/'
AF_OUTPUT_PATH = '../af3/af_output/'

# Structures directory
STRUCTURES_PATH = '../data/structures/'

# Data directories
DATA_PATH = '../data/'
RESIDUES_DIR = 'lists_of_residues/'
CHAIN_LENGTHS_DIR = 'chain_lengths/'
DCCM_MAPS_DIR = 'dccm_maps/'
TORCH_DIR = 'torch/'

# Original ANTIPASTI data directory
OLD_DATA_PATH = '../OLD/data/'

### Loading database

#### exploration

In [4]:
# SARS-CoV-2 spike ectodomain structure (open state)
s6vyb = "MGILPSPGMPALLSLVSLLSVLLMGCVAETGTQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNEVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPSGAGSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKGSGRENLYFQGGGGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGHHHHHHHH"

# Structure of the SARS-CoV-2 spike glycoprotein (closed state)
s6vxx = "MGILPSPGMPALLSLVSLLSVLLMGCVAETGTQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPSGAGSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKGSGRENLYFQGGGGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGHHHHHHHH"

# Structure of post fusion core of 2019-nCoV S2 subunit
s6lxt = "GVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVESGGRGGPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYGG"

In [ ]:
print(len(s6vyb))
print(len(s6vxx))
print(len(s6lxt))


In [ ]:
from antipasti.utils.biology_utils import antibody_sequence_identity
antibody_sequence_identity(s6vyb, s6vxx)

In [ ]:
for idx, (a, b) in enumerate(zip(s6vyb, s6vxx)):
    if a != b:
        print(f"Index: {idx}, s6vyb: {a}, s6vxx: {b}")

In [ ]:
print(antibody_sequence_identity(s6lxt, s6vxx))
print(antibody_sequence_identity(s6lxt, s6vyb))

In [9]:
prot009 = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPSGAGSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCSCGSCCKFDEDDSEPVLKGVKLHYYT"

In [ ]:
print(antibody_sequence_identity(s6vyb, prot009))
print(antibody_sequence_identity(s6vxx, prot009))
print(antibody_sequence_identity(s6lxt, prot009))

In [ ]:
print(len(prot009))

In [ ]:
s_rbd = prot009[319-1:541]
print(len(s_rbd))
print(s_rbd)

In [ ]:
indices = [i for i in range(len(s6vyb)) if s6vyb[i] != s6vxx[i]]
print(indices)

In [14]:
# Antigen sequences taken from antibody PDB structures

s7bz5 = "RVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFHHHHHH"
s7c01 = "RVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFHHHHHH"
s7bwj = "RVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKHHHHHH"

In [ ]:
print(antibody_sequence_identity(s_rbd, s7bz5))
print(antibody_sequence_identity(s_rbd, s7c01))
print(antibody_sequence_identity(s_rbd, s7bwj))

In [ ]:
s_rbd in s7bz5

In [ ]:
s7bwj[:-6] in s7c01

In [ ]:
len(s7bz5)

In [ ]:
len(s7bwj)

In [ ]:
541 - 319 + 1

#### importing database

In [60]:
import os
import pandas as pd

In [61]:
# # Test sequence 1hh6

# light_sequence = "DIKMTQSPSSMYTSLGERVTITCKASQDINSFLTWFLQKPGKSPKTLIYRANRLMIGVPSRFSGSGSGQTYSLTISSLEYEDMGIYYCLQYDDFPLTFGAGTKLDLKRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKEINVKWKIDGSERQNGVLDSWTEQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC"
# heavy_sequence = "QDQLQQSGAELVRPGASVKLSCKALGYIFTDYEIHWVKQTPVHGLEWIGGIHPGSSGTAYNQKFKGKATLTADKSSTTAFMELSSLTSEDSAVYYCTRKDYWGQGTLVTVSAAKTTAPSVYPLVPVCGGTTGSSVTLGCLVKGYFPEPVTLTWNSGSLSSGVHTFPALLQSGLYTLSSSVTVTSNTWPSQTITCNVAHPASSTKVDKKIEPRV"
# antigen_sequence = "DATPEDLGARL"

# sequences = [heavy_sequence + ":" + light_sequence + ":" + antigen_sequence]# load the sequences, making sure you just have heavy_chain:light_chain
# pdb_ids = ["1hh6"]

In [62]:
# # Test sequence no Ag, no structure

# pdb_ids.append("00h4")

# vh_chain = "QVQLVQSGAEVKKPGASVKVSCKASGYTFTGYYMHWVRQAPGQGLEWMGRINPNSGGTNYAQKFQGRVTMTRDTSISTAYMELSRLRSDDTAVYYCARVPYCSSTSCHRDWYFDLWGRGTLVTVSS"
# vl_chain = "DIQMTQSPLSLPVTPGEPASISCRSSQSLLDSDDGNTYLDWYLQKPGQSPQLLIYTLSYRASGVPDRFSGSGSGTDFTLKISRVEAEDVGVYYCMQRIEFPLTFGGGTKVEIK"

# chain = vh_chain + ":" + vl_chain
# sequences.append(chain)


In [63]:
# # Test sequence with 3 Ags
# pdb_ids.append("3lqa")
# chainC = "KKVVLGKKGDTVELTCTASQKKSIQFHWKNSNQIKILGNQGSFLTKGPSKLNDRADSRRSLWDQGNFPLIIKNLKIEDSDTYICEVEDQKEEVQLLVFGLTANSDTHLLQGQSLTLTLESPPGSSPSVQCRSPRGKNIQGGKTLSVSQLELQDSGTWTCTVLQNQKKVEFKIDIVVLAFQKAIDGRHHHHHH"
# chainG = "EIVLENVIENFNMWKNDMVDQMHQDIISLWDQSLKPCVKLTPLCVGAGNCNTSTIAQACPKVSFDPIPIHYCAPAGYAILKCNDKTFNGIGPCNNVSTVQCTHGIKPVVSTQLLLNGSLAEEEVVIRSENISNNVKTIIVHLTESVNITCIGAGHCNINEKAWNETLKKVVEKLVKYFPNKTIEFAPPVGGDLEITTHSFNCGGEFFYCNTTKLFNSIHNSTDSTVNSTDSTAETGNSTNTNITLPCRIRQIINMWQEVGRAMYAPPSKGNITCISDITGLLLTRDGGENKTENNDTEIFRPGGGDMKDNWRSELYKYKVVEIKSGHHHHHH"
# chainH = "QVQVVQSGAEVRKPGASVKVSCKVSGFTLTGLSIHWVRQAPGKGLEWMGGFGPEENEIIYAQKFQGRVSMTEDTSTNTAYMELSSLRSEDTAVYYCATGGNYYNLWTGYYPLAYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDKT"
# chainL = "QSVLTQPPSVSAAPGQKVTISCSGSSSNIGKNYVSWYQQLPGAAPKLLIFDDTQRPSGIPDRFSGSKSGTSATLAITGLQTGDEADYYCGTWDSSLSTGQLFGGGTKLTVLGQPKAAPSVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADSSPVKAGVETTTPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTMAHAECS"

# chain = chainC + ":" + chainG + ":" + chainH + ":" + chainL
# sequences.append(chain)


In [64]:
# Ab-Cov

# Directory containing the .csv files
csv_dir = '../databases/Ab-Cov'

# List to hold individual dataframes
dataframes = []

# Iterate over files in the directory
for filename in os.listdir(csv_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(csv_dir, filename)
        if filename.startswith('covabseq'):
            df = pd.read_csv(file_path)
            dataframes.append(df)

# Concatenate all dataframes into one
df_abcov_all = pd.concat(dataframes, ignore_index=True)


In [ ]:
print(df_abcov_all.shape)
df_abcov_all.columns

In [ ]:
# Ab-Cov virus sequence
prot009 = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPSGAGSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCSCGSCCKFDEDDSEPVLKGVKLHYYT"
s_rbd = prot009[319-1:541]
print(len(s_rbd))


### Producing summary dataframe

In [ ]:
# Select the required columns
df_abcov_summary = df_abcov_all[['Entry', 'Antibody name', 'Neutralizes', 'Viral protein:epitope', 'Binding Affinity (KD)', 'Structures', 'VH sequence', 'VL sequence']]

# Rename the columns
df_abcov_summary.columns = ['entry', 'ab_name', 'virus', 'antigen_type', 'affinity', 'pdb', 'VH', 'VL']

df_abcov_summary['entry'] = df_abcov_summary['entry'].str[-4:]

df_abcov_summary = df_abcov_summary.dropna(subset=['affinity']).sort_values(by='entry')
print(df_abcov_summary.shape)

df_abcov_summary = df_abcov_summary[~(df_abcov_summary['VH'] == 'ND')]
df_abcov_summary = df_abcov_summary[~(df_abcov_summary['VL'] == 'ND')]
print(df_abcov_summary.shape)

df_abcov_summary.reset_index(drop=True)



In [68]:
def convert_affinity(value):
    try:
        return float(value) / 10**9
    except ValueError:
        if '>' in value:
            return 501 / 10**9
        elif '<' in value:
            return 0.0009 / 10**9
        elif '-' in value:
            a, b = map(float, value.split('-'))
            return ((a + b) / 2) / 10**9
        else:
            raise ValueError(f"Unexpected format for affinity value: {value}")

df_abcov_summary['affinity'] = df_abcov_summary['affinity'].apply(convert_affinity)
df_abcov_summary.reset_index(drop=True, inplace=True)


In [ ]:
# df_abcov_summary['affinity'].describe()
df_abcov_summary[100:130]

In [ ]:
print(df_abcov_summary.shape)
print(df_abcov_summary.columns)


In [ ]:
df_abcov_summary['antigen_type'].value_counts()



In [ ]:
virus_counts = df_abcov_summary['virus'].value_counts().reset_index()
virus_counts.columns = ['Virus', 'Counts']
print(virus_counts)


In [ ]:
df_abcov_summary['pdb'].notna().sum()

#### Loading in ANTIPASTI training data entries

In [35]:
import pandas as pd

df_antipasti_train = pd.read_csv('../OLD/data/sabdab_summary_all.tsv', sep='\t', header=0)[['pdb', 'antigen_type', 'affinity']]

#### Using SARS-Cov-2 entries only

In [ ]:
# # Only keeping the SARS-CoV-2 virus antibodies

df_abcov_sarscov2_summary = df_abcov_summary[df_abcov_summary['virus'].str.contains('SARS-CoV-2')].reset_index(drop=True)
print(df_abcov_sarscov2_summary.shape)
df_abcov_sarscov2_summary.head()

In [ ]:
df_abcov_sarscov2_summary['antigen_type']

In [ ]:
# # Create masks for non-alphabet characters or NaNs in 'VH' and 'VL'
# mask_vh = df_abcov_sarscov2_summary['VH'].notna() & ~df_abcov_sarscov2_summary['VH'].str.match(r'^[A-Z]+$', na=False)
# mask_vl = df_abcov_sarscov2_summary['VL'].notna() & ~df_abcov_sarscov2_summary['VL'].str.match(r'^[A-Z]+$', na=False)

# # Combine masks and filter the DataFrame
# invalid_sequences = df_abcov_sarscov2_summary[mask_vh | mask_vl]
# invalid_sequences

In [ ]:
# invalid_sequences.loca[250]['VL']

In [ ]:
# df_abcov_sarscov2_summary['VH'] = df_abcov_sarscov2_summary['VH'].str.replace(' ','', regex=True).str.upper()
# df_abcov_sarscov2_summary['VL'] = df_abcov_sarscov2_summary['VL'].str.replace(' ','', regex=True).str.upper()

In [ ]:
# print(df_abcov_sarscov2_summary['antigen_type'].value_counts())

In [ ]:
# df_abcov_sarscov2_summary.to_csv(os.path.join('../data',summary_file_name), sep='\t', index=False)

#### Using S: RBD entries only

In [ ]:
# Only keeping the S:RBD virus antibodies

df_abcov_srbd_summary = df_abcov_summary[df_abcov_summary['antigen_type'] == 'S: RBD'].reset_index(drop=True)
df_abcov_srbd_summary = df_abcov_srbd_summary[df_abcov_srbd_summary['virus'].str.contains('SARS-CoV-2')].reset_index(drop=True)
print(df_abcov_srbd_summary.shape)
df_abcov_srbd_summary.head()

In [ ]:
# Create masks for non-alphabet characters or NaNs in 'VH' and 'VL'
mask_vh = df_abcov_srbd_summary['VH'].notna() & ~df_abcov_srbd_summary['VH'].str.match(r'^[A-Z]+$', na=False)
mask_vl = df_abcov_srbd_summary['VL'].notna() & ~df_abcov_srbd_summary['VL'].str.match(r'^[A-Z]+$', na=False)

# Combine masks and filter the DataFrame
invalid_sequences = df_abcov_srbd_summary[mask_vh | mask_vl]
print(invalid_sequences)

# Also remove spaces
df_abcov_srbd_summary['VH'] = df_abcov_srbd_summary['VH'].str.replace(' ','', regex=True).str.upper()
df_abcov_srbd_summary['VL'] = df_abcov_srbd_summary['VL'].str.replace(' ','', regex=True).str.upper()

In [ ]:
print(df_abcov_srbd_summary['virus'].value_counts())

In [ ]:
# Get the non-NA values from df_abcov_summary['pdb']
non_na_pdb_abcov = df_abcov_summary['pdb'].dropna()

print(non_na_pdb_abcov.shape[0])

In [ ]:
# Check for duplicates in VH column#
vh_duplicates = df_abcov_srbd_summary[df_abcov_srbd_summary.duplicated(subset=['VH'], keep=False)]
print(f"Number of duplicate VH sequences: {len(vh_duplicates)}")
if len(vh_duplicates) > 0:
    print("\nDuplicate VH sequences:")
    print(vh_duplicates[['VH']].sort_values('VH'))

In [ ]:
vl_duplicates = df_abcov_srbd_summary[df_abcov_srbd_summary.duplicated(subset=['VL'], keep=False)].dropna()
print(f"\nNumber of duplicate VL sequences: {len(vl_duplicates)}")
if len(vl_duplicates) > 0:
    print("\nDuplicate VL sequences:")
    print(vl_duplicates[['VL']].sort_values('VL'))

In [ ]:
# Check if any of these values are in df_antipasti_train['pdb']
non_na_pdb_abcov.isin(df_antipasti_train['pdb']).any()

In [42]:
summary_file_name = "abcov_srbd_summary.tsv"

df_abcov_srbd_summary.to_csv(os.path.join('../data',summary_file_name), sep='\t', index=False)

### Getting lists of residues and producing AF3 inputs

In [43]:
from anarci import run_anarci
import numpy as np
import os
import json

In [11]:
def sequences_to_afinput(name, sequences_dict, save_path, file_name = None):
    if file_name is None:
        file_name = name
    file_name += '.json'
    input_data = {
        "name": name,
        "modelSeeds": [1],
        "sequences": [
            {"protein": {"id": code, "sequence": seq}}
            for code, seq in sequences_dict.items()
        ],
        "dialect": "alphafold3",
        "version": 1
    }
    
    with open(os.path.join(save_path, file_name), 'w') as json_file:
        json.dump(input_data, json_file, indent=4)

In [12]:
def sequences_to_residues(entries, sequences, tag='', residues_path = os.path.join(DATA_PATH,RESIDUES_DIR), af_input_path = AF_INPUT_PATH, get_af_inputs = True):

    residues_dict_anarci = {}
    
    for i, seq in enumerate(sequences):

        try:
            split_seq = seq.split(':')
        except Exception as e:
            print(f"Error: {e}")
            print(f"Entry: {entries[i]}")
            print(f"Sequence: {seq}")
            return

        
        idx_chaintype = {'VH': None, 'VL': None, 'Ag': []}
        residues = []

        for j, sub_seq in enumerate(split_seq):
            results = run_anarci([(f'chain{j}', sub_seq)], scheme='chothia')

            # Checking which chain is heavy/light/antigen
            if results[2][0]:
                anarci_chain = results[1][0][0][0]
                if anarci_chain[-1][0][0] > 107 and  anarci_chain[-1][0][0] <= 113:            
                    chaintype = 'VH'
                    res_code = 'A'
                elif anarci_chain[-1][0][0] <= 107:
                    chaintype = 'VL'
                    res_code = 'B'
                else:
                    print(f"Sequence: {anarci_chain}")
                    raise ValueError(f"Unknown chain type for sequence {j} of {entries[i]}")
            else:
                chaintype = 'Ag'
                res_code = chr(ord('C') + len(idx_chaintype['Ag']))  # Assign C, D, E, ... for antigens
                    
            if chaintype == 'Ag':
                idx_chaintype[chaintype].append(j)
            else:
                if idx_chaintype[chaintype]:
                    raise ValueError(f"Multiple {chaintype} chains for antibody {entries[i]}")
                idx_chaintype[chaintype] = j

            res_list = []
            if chaintype == 'VH' or chaintype == 'VL':
                for pos, residue in anarci_chain:
                    if residue != '-':
                        full_res = f'{residue}{res_code}{str(pos[0]).rjust(3)}'
                        if pos[1].strip():
                            full_res += pos[1].strip()
                        else:
                            full_res += ' '
                        res_list.append(full_res)
            elif chaintype == 'Ag':
                for idx, residue in enumerate(sub_seq):
                    pos = idx + 1
                    full_res = f'{residue}{res_code}{str(pos).rjust(3)} '
                    res_list.append(full_res)
            else:
                raise ValueError("!!!") # If chaintype isn't VH, VL, or Ag something has gone wrong
            residues.append(res_list)

        if get_af_inputs:
            sequences_to_afinput(
                entries[i],
                {res_code : split_seq[idx_chaintype[chaintype]] for chaintype, res_code in [('VH','A'), ('VL','B')] if idx_chaintype[chaintype] is not None} |
                {chr(ord('C') + k): split_seq[j] for k, j in enumerate(idx_chaintype['Ag'])},
                af_input_path,
                file_name = tag+entries[i]
            )

        list_of_residues = ['START-Ab']
        if idx_chaintype['VH'] is not None:
            list_of_residues += residues[idx_chaintype['VH']]
        if idx_chaintype['VL'] is not None:
            list_of_residues += residues[idx_chaintype['VL']]
        list_of_residues += ['END-Ab']
        for k, j in enumerate(idx_chaintype['Ag']):
            list_of_residues += residues[j]
        residues_dict_anarci[entries[i]] = list_of_residues

    # Saving ANARCI residues
    for entry in entries:
        save_filename = f'{entry}.npy'
        np.save(os.path.join(residues_path,save_filename), residues_dict_anarci[entry])
    
    return residues_dict_anarci

In [70]:
# sequences_to_residues(pdb_ids, sequences, tag='test', get_af_inputs=False)

In [71]:
# df_abcov_summary = df_abcov_summary[~df_abcov_summary['entry'].isin(['0178', '0186'])] # removing issues

In [30]:
entries = df_abcov_srbd_summary['entry'].tolist()
sequences = (df_abcov_srbd_summary['VH'] + ':' + df_abcov_srbd_summary['VL'].fillna('') + ':' + s_rbd).tolist()
sequences = [seq.replace('::', ':') for seq in sequences]
residues_dict_anarci = sequences_to_residues(entries, sequences, tag='AbCov', residues_path=os.path.join(DATA_PATH, RESIDUES_DIR), get_af_inputs = True)

In [ ]:
num_files_af_input = len(os.listdir(AF_INPUT_PATH))
print(f"Number of files in {AF_INPUT_PATH}: {num_files_af_input}")

num_files_residues = len(os.listdir(os.path.join(DATA_PATH, RESIDUES_DIR)))
print(f"Number of files in {RESIDUES_DIR}: {num_files_residues}")

In [ ]:
df_abcov_srbd_summary[df_abcov_srbd_summary['entry']=='1845']

In [ ]:
residues_dict_anarci['1845']

In [ ]:
import numpy as np
import os


sample_residues_list = np.load(os.path.join(DATA_PATH, RESIDUES_DIR, '0796.npy'))
print(sample_residues_list)
print(len(sample_residues_list))

#### Comparing to old data

In [ ]:
import numpy as np
import os

residues_path = '../OLD/data/lists_of_residues/'
residues_dict_old = {}

for filename in os.listdir(residues_path):
    file_path = os.path.join(residues_path, filename)
    residues = np.load(file_path)
    filename_without_ext = os.path.splitext(filename)[0]
    residues_dict_old[filename_without_ext] = residues




In [ ]:
# List of residues comparison

id_check = '3lqa'

print("(ANARCI, OLD)")

for res_anarci, res_old in zip(residues_dict_anarci[id_check], residues_dict_old[id_check]):
    res_anarci = str(res_anarci)
    res_old = str(res_old)
    to_print = f"({res_anarci}, {res_old})"
    if res_anarci[0] != res_old[0] or res_anarci[2:] != res_old[2:]:
        to_print += "!!!"
    print(to_print)


In [30]:
# list_of_residues = list(residues_dict_old[pdb_ids[0]])[1:]

# chain_pos = 0
# if 'END-Ab' in list_of_residues:
#     list_of_residues = list_of_residues[:list_of_residues.index('END-Ab')]
#     chain_pos = 1
# else:
#     list_of_residues = residues_dict_old[pdb_ids[0]][:-1]

# h_chain = list_of_residues[0][chain_pos]
# l_chain = list_of_residues[-1][chain_pos]

# print(len([idx for idx in list_of_residues if idx[chain_pos] == h_chain]))
# if h_chain != l_chain:
#     print(len([idx for idx in list_of_residues if idx[chain_pos] == l_chain]))
# else:
#     print(0)

In [ ]:
# selected_entries = np.load("../OLD/data/chain_lengths/selected_entries.npy")
# heavy_lengths = np.load("../OLD/data/chain_lengths/heavy_lengths.npy")
# light_lengths = np.load("../OLD/data/chain_lengths/light_lengths.npy")


# pdb_idx = np.where(selected_entries == pdb_ids[0])[0][0]
# print(heavy_lengths[pdb_idx], light_lengths[pdb_idx])

### Getting AF3 outputs

First run AF3 using the input json files produced from the previous step to produce the output Computed Structure Models.

#### Checking confidence metrics

In [ ]:
# import json
# import pandas as pd
# import numpy as np

# # Define the directory containing AlphaFold output structures
# af3_structures_dir = AF_OUTPUT_PATH

# # Initialize a list to store summary data
# all_summary_confidences = []

# # Iterate through each protein directory
# for protein_name in os.listdir(af3_structures_dir):
#     # Construct the path to the summary_confidences.json file
#     summary_confidences_path = os.path.join(af3_structures_dir, protein_name, f"{protein_name}_summary_confidences.json")
#     confidences_path = os.path.join(af3_structures_dir, protein_name, f"{protein_name}_confidences.json")

    
#     # Check if the file exists
#     if os.path.isfile(summary_confidences_path) and os.path.isfile(confidences_path):
#         print(f"Processing: {protein_name}")
        
#         # Load the JSON file
#         with open(summary_confidences_path, 'r') as f:
#             summary_confidence_data = json.load(f)
        
#         # Extract relevant confidence metrics
#         protein_summary = {"entry": protein_name, **summary_confidence_data}
        
#         # Load the JSON file
#         with open(confidences_path, 'r') as f:
#             confidence_data = json.load(f)

#         # Add these metrics to the protein summary
#         protein_summary['mean_plddt'] = np.mean(confidence_data['atom_plddts'])
#         protein_summary['mean_plddt_abonly'] = np.mean(confidence_data['atom_plddts'][:-223])
#         protein_summary['max_pae'] = np.max(confidence_data['pae'])
#         protein_summary['max_pae_abonly'] = np.max(confidence_data['pae'][:-223])
    
#         # Append the summary to the list
#         all_summary_confidences.append(protein_summary)
#         if protein_name == '1718':
#             print(protein_summary)
#             print(all_summary_confidences[-2:-1])

# # Convert the summary data into a pandas DataFrame for easier analysis
# df_confidence_summary = pd.DataFrame(all_summary_confidences)
# df_confidence_summary = df_confidence_summary.sort_values(by='entry').reset_index(drop=True)


In [ ]:
import json
import pandas as pd
import numpy as np

# Define the directory containing AlphaFold output structures
af3_structures_dir = AF_OUTPUT_PATH

# Initialize a list to store summary data
all_summary_confidences = []

# Iterate through each protein directory
for protein_name in os.listdir(af3_structures_dir):
    # Construct the path to the summary_confidences.json file
    summary_confidences_path = os.path.join(af3_structures_dir, protein_name, f"{protein_name}_summary_confidences.json")
    confidences_path = os.path.join(af3_structures_dir, protein_name, f"{protein_name}_confidences.json")

    
    # Check if the file exists
    if os.path.isfile(summary_confidences_path) and os.path.isfile(confidences_path):
        print(f"Processing: {protein_name}")
        
        # Load the JSON file
        with open(summary_confidences_path, 'r') as f:
            summary_confidence_data = json.load(f)
        
        # Extract relevant confidence metrics
        protein_summary = {"entry": protein_name, **summary_confidence_data}
        
        # Load the JSON file
        with open(confidences_path, 'r') as f:
            confidence_data = json.load(f)

        # Add these metrics to the protein summary
        atom_chain_ids = np.array(confidence_data['atom_chain_ids'])
        heavy_indices = np.where(atom_chain_ids == 'A')[0]
        light_indices = np.where(atom_chain_ids == 'B')[0]
        antigen_indices = np.where(atom_chain_ids == 'C')[0]
        atom_plddts = np.array(confidence_data['atom_plddts'])
        protein_summary['plddts'] = atom_plddts.tolist()
        protein_summary['mean_plddt'] = np.mean(atom_plddts)
        protein_summary['mean_plddt_abonly'] = np.mean(atom_plddts[np.concatenate((heavy_indices, light_indices))])
        protein_summary['mean_plddt_agonly'] = np.mean(atom_plddts[antigen_indices])

        # protein_summary['mean_plddt_abonly'] = np.mean(confidence_data['atom_plddts'][:-223])
        pae_array = np.array(confidence_data['pae'])
        protein_summary['paes'] = pae_array.tolist()
        protein_summary['max_pae'] = np.max(pae_array)
        protein_summary['max_pae_abonly'] = np.max(pae_array[:-223])
        protein_summary['max_pae_ababonly'] = np.max(pae_array[:-223,:-223])
        protein_summary['max_pae_agonly'] = np.max(pae_array[-223:,-223:])
    
        # Append the summary to the list
        all_summary_confidences.append(protein_summary)
        if protein_name == '1718':
            print(protein_summary)
            print(all_summary_confidences[-2:-1])

# Convert the summary data into a pandas DataFrame for easier analysis
df_confidence_summary = pd.DataFrame(all_summary_confidences)
df_confidence_summary = df_confidence_summary.sort_values(by='entry').reset_index(drop=True)


In [ ]:
print(len(confidence_data['token_res_ids']))
print(len(confidence_data['token_chain_ids']))
print(len(confidence_data['pae']))
print(len(confidence_data['atom_plddts']))

In [39]:
# selected_entries = np.load("../data/chain_lengths/selected_entries.npy")
# heavy_lengths = np.load("../data/chain_lengths/heavy_lengths.npy")
# light_lengths = np.load("../data/chain_lengths/light_lengths.npy")
# idx_selected = np.where(selected_entries == '0796')[0]
# print(heavy_lengths[idx_selected], light_lengths[idx_selected])

# print(len(confidence_data['pae']))

In [ ]:
df_confidence_summary.head()

In [ ]:
df_confidence_summary.describe()

In [ ]:
len(df_confidence_summary['plddts'][0])

In [ ]:
df_confidence_summary['chain_pair_pae_min'][0]

In [ ]:
df_confidence_summary[df_confidence_summary['entry'] == '1718']

In [182]:
df_confidence_summary.to_csv(os.path.join(DATA_PATH, 'confidence/af3_srbd_confidence_summary.csv'), index=False)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 5))

ax1.hist(df_confidence_summary['mean_plddt'], bins=20, color='red', alpha=0.7)
ax1.set_title('Distribution of mean_plddt')
ax1.set_xlabel('mean_plddt')
ax1.set_ylabel('Frequency')

ax2.hist(df_confidence_summary['max_pae'], bins=20, color='purple', alpha=0.7)
ax2.set_title('Distribution of max_pae')
ax2.set_xlabel('max_pae')
ax2.set_ylabel('Frequency')

ax3.hist(df_confidence_summary['max_pae_abonly'], bins=20, color='blue', alpha=0.7)
ax3.set_title('Distribution of max_pae_abonly')
ax3.set_xlabel('max_pae_abonly')
ax3.set_ylabel('Frequency')

# ax3.hist(df_confidence_summary['ptm'], bins=20, color='blue', alpha=0.7)
# ax3.set_title('Distribution of ptm')
# ax3.set_xlabel('ptm')
# ax3.set_ylabel('Frequency')

# ax4.hist(df_confidence_summary['iptm'], bins=20, color='green', alpha=0.7)
# ax4.set_title('Distribution of iptm')
# ax4.set_xlabel('iptm')
# ax4.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scienceplots

# Set style
plt.style.use(['science', 'nature', 'no-latex'])

# Create figure with correct panel organization
fig = plt.figure(figsize=(10, 8), dpi=300)
gs = fig.add_gridspec(2, 2)
ax1 = fig.add_subplot(gs[0, 0])  # Top-left: mean pLDDT (entire)
ax2 = fig.add_subplot(gs[0, 1])  # Top-right: max PAE (entire)
ax3 = fig.add_subplot(gs[1, 0])  # Bottom-left: mean pLDDT (antibody-only)
ax4 = fig.add_subplot(gs[1, 1])  # Bottom-right: max PAE (antibody-only)

# --- Configure bins and ranges ---
# pLDDT range 75-93
plddt_bins = np.linspace(75, 93, 19)  # 18 bins
plddt_ticks = np.arange(75, 94, 3)    # Ticks every 3 units

# PAE range 28-32
pae_bins = np.linspace(17, 32, 17)    # 16 bins
pae_ticks = np.arange(17, 33, 1)      # Ticks every 1 unit

# --- Calculate maximum frequencies ---
plddt_max = max(
    np.histogram(df_confidence_summary['mean_plddt'], bins=plddt_bins)[0].max(),
    np.histogram(df_confidence_summary['mean_plddt_abonly'], bins=plddt_bins)[0].max()
) * 1.15

pae_max = max(
    np.histogram(df_confidence_summary['max_pae'], bins=pae_bins)[0].max(),
    np.histogram(df_confidence_summary['max_pae_abonly'], bins=pae_bins)[0].max()
) * 1.15

# --- Plot 1: mean pLDDT (entire) ---
ax1.hist(df_confidence_summary['mean_plddt'], 
         bins=plddt_bins, color='royalblue', alpha=0.7, 
         edgecolor='k', linewidth=0.5)
ax1.set(xlabel='Mean pLDDT (entire structure)', 
        ylabel='Count',
        xlim=(75, 93),
        ylim=(0, plddt_max),
        xticks=plddt_ticks)

# --- Plot 2: max PAE (entire) ---
ax2.hist(df_confidence_summary['max_pae'], 
         bins=pae_bins, color='indianred', alpha=0.7,
         edgecolor='k', linewidth=0.5)
ax2.set(xlabel='Max PAE (entire structure)',
        ylabel='Count',
        ylim=(0, pae_max),
        xlim=(17, 32),
        xticks=pae_ticks)

# --- Plot 3: mean pLDDT (antibody-only) ---
ax3.hist(df_confidence_summary['mean_plddt_abonly'], 
         bins=plddt_bins, color='dodgerblue', alpha=0.7,
         edgecolor='k', linewidth=0.5)
ax3.set(xlabel='Mean pLDDT (antibody only)',
        ylabel='Count',
        xlim=(75, 93),
        ylim=(0, plddt_max),
        xticks=plddt_ticks)

# --- Plot 4: max PAE (antibody-only) ---
ax4.hist(df_confidence_summary['max_pae_abonly'], 
         bins=pae_bins, color='crimson', alpha=0.7,
         edgecolor='k', linewidth=0.5)
ax4.set(xlabel='Max PAE (antibody only)',
        ylabel='Count',
        xlim=(17, 32),
        ylim=(0, pae_max),
        xticks=pae_ticks)

# --- Consistent styling ---
for ax in [ax1, ax2, ax3, ax4]:
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.grid(alpha=0.2)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlabel(ax.get_xlabel(), fontsize=12) 
    ax.set_ylabel(ax.get_ylabel(), fontsize=12)

plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scienceplots

# Set style
plt.style.use(['science', 'nature', 'no-latex'])

fontsize = 20
legendsize=16

# Create figure with two panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=300)

# --- Configure bins and ranges ---
# pLDDT range
plddt_bins = np.linspace(75, 93, 19)
plddt_ticks = np.arange(75, 94, 3)

# PAE range 
pae_bins = np.linspace(28, 32, 21)
pae_ticks = np.arange(28, 33, 1)
# pae_bins = np.linspace(17, 32, 21)
# pae_ticks = np.arange(17, 33, 1)

# --- Plot 1: Combined pLDDT (entire + antibody-only) ---
ax1.hist([df_confidence_summary['mean_plddt'], df_confidence_summary['mean_plddt_abonly']],
         bins=plddt_bins, 
         color=['royalblue', 'indianred'], 
         alpha=0.7,
         edgecolor='k', 
         linewidth=0.5,
         label=['Entire structure', 'Excluding antigen'])

ax1.set(xlabel='Mean pLDDT',
        ylabel='Count',
        xlim=(75, 93),
        xticks=plddt_ticks)
ax1.legend(fontsize=legendsize, frameon=False)

# --- Plot 2: Combined PAE (entire + antibody-only) ---
ax2.hist([df_confidence_summary['max_pae'], df_confidence_summary['max_pae_abonly']],
         bins=pae_bins,
         color=['royalblue', 'indianred'],
         alpha=0.7,
         edgecolor='k',
         linewidth=0.5,
         label=['Entire structure', 'Excluding antigen'])
# ax2.hist([df_confidence_summary['max_pae'], df_confidence_summary['max_pae_abonly'], df_confidence_summary['max_pae_ababonly']],
#          bins=pae_bins,
#          color=['royalblue', 'indianred', 'purple'],
#          alpha=0.7,
#          edgecolor='k',
#          linewidth=0.5,
#          label=['Entire structure', 'Excluding antigen'])

ax2.set(xlabel='Max PAE',
        ylabel='Count',
        xlim=(28, 32),
        xticks=pae_ticks)
ax2.legend(fontsize=legendsize, frameon=False)

# --- Consistent styling ---
for ax in [ax1, ax2]:
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.grid(alpha=0.2)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlabel(ax.get_xlabel(), fontsize=fontsize)
    ax.set_ylabel(ax.get_ylabel(), fontsize=fontsize)

plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
import seaborn as sns

import matplotlib.pyplot as plt

# Select the columns of interest
columns_of_interest = ['mean_plddt', 'max_pae']
df_correlation = df_confidence_summary[columns_of_interest]

# Create a pairplot to visualize the correlation
sns.pairplot(df_correlation, kind='scatter', diag_kind='kde', plot_kws={'alpha': 0.7})
plt.suptitle('Correlation between mean_plddt, max_pae, and max_pae_abonly', y=1.02)
plt.show()

#### Converting to ANTIPASTI pdb input

In [30]:
import pymol2
import os

In [ ]:
cif_in_dir = AF_OUTPUT_PATH
pdb_out_dir = STRUCTURES_PATH

# List all folders in the directory
files = [f for f in os.listdir(cif_in_dir) if os.path.isdir(os.path.join(cif_in_dir, f))]

# Print the number of files
print(f"Number of files in {cif_in_dir}: {len(files)}")

In [ ]:
# iterate over files in
# that directory
for protein_name in os.listdir(cif_in_dir):
    cifname = os.path.join(cif_in_dir, protein_name, protein_name+'_model.cif')
    if os.path.isfile(cifname):
        print(cifname)
        with pymol2.PyMOL() as pymol:
            pymol.cmd.load(cifname,'myprotein')
            pymol.cmd.save(os.path.join(pdb_out_dir, protein_name+'_af.pdb'), selection='myprotein')

In [ ]:
# List all files in the directory
files = [f for f in os.listdir(pdb_out_dir)]

# Print the number of files
print(f"Number of files in {pdb_out_dir}: {len(files)}")